# API Usage Examples with Python

This Jupyter Notebook provides some illustrative examples on how to interact with the API using Python.

## Overview

In this notebook, we will:
1. Explore available graphs.
2. Investigate entities and relationships within a specific graph.
3. Answer some interesting questions using the API.

In [8]:
import requests

# CONFIG
base_url = "http://localhost:8000" # if you are running the server locally

Let's start by taking a look at the available graphs. The endpoint `/graphs` provides a list of all available graphs:

In [9]:
def get_graphs():
    endpoint = f"{base_url}/graphs"
    response = requests.get(endpoint)
    return response.json()

get_graphs()

[{'name': 'codexlarge',
  'description': 'The CoDEx large dataset from Safavi and Koutra (2020).',
  'n_entities': 77951,
  'n_triples': 612437,
  'n_relations': 69,
  'dataset_url': 'https://github.com/tsafavi/codex'},
 {'name': 'codexsmall',
  'description': 'The CoDEx small dataset from Safavi and Koutra (2020).',
  'n_entities': 2034,
  'n_triples': 36543,
  'n_relations': 42,
  'dataset_url': 'https://github.com/tsafavi/codex'},
 {'name': 'countries',
  'description': 'The Countries dataset.',
  'n_entities': 271,
  'n_triples': 1158,
  'n_relations': 2,
  'dataset_url': 'https://github.com/ZhenfengLei/KGDatasets/tree/master/Countries'},
 {'name': 'dbpedia50',
  'description': 'Graph containing structured information extracted from Wikipedia articles.',
  'n_entities': 24624,
  'n_triples': 34421,
  'n_relations': 351,
  'dataset_url': 'https://github.com/ZhenfengLei/KGDatasets/tree/master/DBpedia50'},
 {'name': 'nations',
  'description': 'Graph representation of relationships be

For the rest of the examples, we will be using the Nations graph. Run the next cell in order to explore the different entities and relationships within this graph:

In [10]:
def get_entity_list(graph_name):
    endpoint = f"{base_url}/graphs/{graph_name}/entity_list"
    response = requests.get(endpoint)
    return response.json()

def get_relation_list(graph_name):
    endpoint = f"{base_url}/graphs/{graph_name}/relationship_list"
    response = requests.get(endpoint)
    return response.json()

print(get_entity_list("nations"))
print(get_relation_list("nations"))

{'0': 'brazil', '1': 'burma', '2': 'china', '3': 'cuba', '4': 'egypt', '5': 'india', '6': 'indonesia', '7': 'israel', '8': 'jordan', '9': 'netherlands', '10': 'poland', '11': 'uk', '12': 'usa', '13': 'ussr'}
{'0': 'accusation', '1': 'aidenemy', '2': 'attackembassy', '3': 'blockpositionindex', '4': 'booktranslations', '5': 'boycottembargo', '6': 'commonbloc0', '7': 'commonbloc1', '8': 'commonbloc2', '9': 'conferences', '10': 'dependent', '11': 'duration', '12': 'economicaid', '13': 'eemigrants', '14': 'embassy', '15': 'emigrants3', '16': 'expeldiplomats', '17': 'exportbooks', '18': 'exports3', '19': 'independence', '20': 'intergovorgs', '21': 'intergovorgs3', '22': 'lostterritory', '23': 'militaryactions', '24': 'militaryalliance', '25': 'negativebehavior', '26': 'negativecomm', '27': 'ngo', '28': 'ngoorgs3', '29': 'nonviolentbehavior', '30': 'officialvisits', '31': 'pprotests', '32': 'relbooktranslations', '33': 'reldiplomacy', '34': 'releconomicaid', '35': 'relemigrants', '36': 'relex

Now that we have a better idea of the entities and relationships within the graph, you can play with the possibilities of the different endpoints of the API. For some inspiration, you can start with the following simple questions:

- What is the embedding representation of Brazil?
- Which countries are more likely to depend (link=dependent) on the UK?
- How similar are, on average, these two groups of entities:
    - China and Indonesia
    - Poland and Netherlands
- Which entity lies at the center of Indonesia, India, and Cuba?

In [14]:
# q 1
def get_embedding(graph_name, embedding_model, entity):
    endpoint = f"{base_url}/embeddings/by-entity/{graph_name}/{embedding_model}/{entity}"
    response = requests.get(endpoint)
    return response.json()

get_embedding("nations","transe","brazil")

{'entity_name': 'brazil',
 'graph_name': 'nations',
 'embedding': [0.3558332324028015,
  -0.5394994616508484,
  0.16455808281898499,
  0.695117175579071,
  0.2684321701526642],
 'embedding_model': 'transe'}

In [23]:
# q 2
def predict_tail(graph_name, embedding_model, head, rel, k):
    endpoint = f"{base_url}/predictions/entity/{graph_name}/{embedding_model}/{head}/{rel}"
    response = requests.get(endpoint,params={"k": k})
    return response.json()

predict_tail("nations","transe","uk","dependent",k=10)

[{'head': 'uk',
  'relationship': 'dependent',
  'tail': 'uk',
  'score': -1.760583519935608},
 {'head': 'uk',
  'relationship': 'dependent',
  'tail': 'india',
  'score': -2.1559128761291504},
 {'head': 'uk',
  'relationship': 'dependent',
  'tail': 'ussr',
  'score': -2.486320734024048},
 {'head': 'uk',
  'relationship': 'dependent',
  'tail': 'indonesia',
  'score': -2.4956271648406982},
 {'head': 'uk',
  'relationship': 'dependent',
  'tail': 'netherlands',
  'score': -2.8541512489318848},
 {'head': 'uk',
  'relationship': 'dependent',
  'tail': 'burma',
  'score': -2.8986239433288574},
 {'head': 'uk',
  'relationship': 'dependent',
  'tail': 'usa',
  'score': -2.9396278858184814},
 {'head': 'uk',
  'relationship': 'dependent',
  'tail': 'brazil',
  'score': -3.0151565074920654},
 {'head': 'uk',
  'relationship': 'dependent',
  'tail': 'egypt',
  'score': -3.033717155456543},
 {'head': 'uk',
  'relationship': 'dependent',
  'tail': 'poland',
  'score': -3.0530521869659424}]

In [24]:
# q 3
entities_1 = ["china","indonesia"]
entities_2 = ["poland","netherlands"]

def get_similarity(graph_name, embedding_model, metric, entities_1, entities_2):
    endpoint = f"{base_url}/similarities/cosine-similarity/entities/multiple/{graph_name}/{embedding_model}/{metric}"
    data = {
        "entity_list1": entities_1,
        "entity_list2": entities_2
    }
    response = requests.post(endpoint, json=data)
    return response.json()

get_similarity("nations","transe","average",entities_1,entities_2)

{'similarity': -0.27014235034585, 'metric_used': 'average'}

In [25]:
# q 4
entities = ["indonesia", "india", "cuba"]

def get_center(graph_name, embedding_model, entities):
    endpoint = f"{base_url}/centers/{graph_name}/{embedding_model}"
    response = requests.post(endpoint, json=entities)
    return response.json()

get_center("nations","transe", entities)

{'input entities': ['indonesia', 'india', 'cuba'],
 'geometric_median': {'point': [0.0, 0.0, 0.0, 0.0, 1.3704552629442007e-13],
  'radii': {'median': 1.0, 'mean': 1.0, 'max': 1.0},
  'closest entity': 'indonesia'},
 'centroid': {'point': [0.0737074613571167,
   -0.38277509808540344,
   0.08278460800647736,
   -0.2884850800037384,
   0.3207132816314697],
  'radii': {'median': 0.7373396158218384,
   'mean': 0.7938991189002991,
   'max': 1.0088920593261719},
  'closest entity': 'uk'}}